In [ ]:
# 03 - Linear Models
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from src.models.preprocessor import get_preprocessor


In [ ]:
print("="*60)
print("Loading Dataset")
print("="*60)

df = pd.read_parquet("../data/processed/featured_taxi_data.parquet")


In [ ]:
drop_columns = [
    "trip_duration_minutes",
    "tpep_dropoff_datetime",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "tolls_amount",
    "extra",
    "airport_fee",
    "Airport_fee",
    "congestion_surcharge",
    "improvement_surcharge"
]

X = df.drop(columns=drop_columns)
y = df["trip_duration_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5)
}

results = []

print("\nTraining Models...\n")

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", get_preprocessor()),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"{name}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print()
    
    results.append([name, mae, rmse, r2])

results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

print(results)


In [ ]:
# ==========================================================
# Comparison Plot
# ==========================================================

plt.figure(figsize=(8,5))

plt.bar(results["Model"], results["MAE"])

plt.title("Linear Models Comparison")

plt.ylabel("MAE")

plt.xticks(rotation=15)

plt.tight_layout()

plt.show()

print("\nCompleted Successfully.")
